# Book Recommendation System - Collaborative Filtering Pipeline

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load raw datasets
books = pd.read_csv('data/raw/Books.csv', low_memory=False)
ratings = pd.read_csv('data/raw/Ratings.csv')
users = pd.read_csv('data/raw/Users.csv')

# Merge ratings and books to form final_df while preserving original data
final_df = ratings.merge(books, on='ISBN')
print("final_df shape:", final_df.shape)

final_df shape: (1031136, 10)


In [2]:
# 1. User Filtering (Active Users): Filter for users who have rated > 200 books
user_rating_counts = final_df.groupby('User-ID').count()['Book-Rating']
padaku_users = user_rating_counts[user_rating_counts > 200].index

print("Number of active users (>200 ratings):", len(padaku_users))

Number of active users (>200 ratings): 811


In [3]:
# 2. Book Filtering (Popular Books): Filter final_df using active users, keep books with >= 50 ratings
filtered_rating = final_df[final_df['User-ID'].isin(padaku_users)]

book_rating_counts = filtered_rating.groupby('Book-Title').count()['Book-Rating']
famous_books = book_rating_counts[book_rating_counts >= 50].index

final_ratings = filtered_rating[filtered_rating['Book-Title'].isin(famous_books)]
print("final_ratings shape:", final_ratings.shape)

final_ratings shape: (58586, 10)


In [4]:
# 3. Pivot Table Generation
pt = final_ratings.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating')
pt.fillna(0, inplace=True)

# 4. Cosine Similarity Computation
similarity_scores = cosine_similarity(pt)

# Verification prints
print("pt.shape:", pt.shape)
print("np.count_nonzero(similarity_scores):", np.count_nonzero(similarity_scores))

pt.shape: (706, 810)
np.count_nonzero(similarity_scores): 364308


In [5]:
# 5. Recommendation Function
def recommend(book_name):
    if book_name not in pt.index:
        print(f"Book '{book_name}' not found in pivot table.")
        return []
    
    # Find index of the input book
    index = np.where(pt.index == book_name)[0][0]
    
    # Sort similarity scores in descending order (excluding self at index 0)
    similar_items = sorted(list(enumerate(similarity_scores[index])), key=lambda x: x[1], reverse=True)[1:6]
    
    data = []
    for i in similar_items:
        item = []
        temp_df = books[books['Book-Title'] == pt.index[i[0]]]
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Image-URL-M'].values))
        item.append(round(float(i[1]), 4))
        data.append(item)
        
    return data

# Test recommendation function
recommend('1984')

[['Animal Farm', 'George Orwell', 'http://images.amazon.com/images/P/0451526341.01.MZZZZZZZ.jpg', 0.2703], ["The Handmaid's Tale", 'Margaret Atwood', 'http://images.amazon.com/images/P/0449212602.01.MZZZZZZZ.jpg', 0.264], ['Brave New World', 'Aldous Huxley', 'http://images.amazon.com/images/P/0060809833.01.MZZZZZZZ.jpg', 0.2367], ['The Vampire Lestat (Vampire Chronicles, Book II)', 'ANNE RICE', 'http://images.amazon.com/images/P/0345313860.01.MZZZZZZZ.jpg', 0.233], ['The Hours : A Novel', 'Michael Cunningham', 'http://images.amazon.com/images/P/0312243022.01.MZZZZZZZ.jpg', 0.2263]]

In [6]:
# Additional test recommendation
recommend('The Fellowship of the Ring (The Lord of the Rings, Part 1)')

[['The Two Towers (The Lord of the Rings, Part 2)', 'J.R.R. TOLKIEN', 'http://images.amazon.com/images/P/0345339711.01.MZZZZZZZ.jpg', 0.5456], ['The Return of the King (The Lord of the Rings, Part 3)', 'J.R.R. TOLKIEN', 'http://images.amazon.com/images/P/0345339738.01.MZZZZZZZ.jpg', 0.3736], ['Harry Potter and the Prisoner of Azkaban (Book 3)', 'J. K. Rowling', 'http://images.amazon.com/images/P/0439136350.01.MZZZZZZZ.jpg', 0.2793], ['Harry Potter and the Goblet of Fire (Book 4)', 'J. K. Rowling', 'http://images.amazon.com/images/P/0439139597.01.MZZZZZZZ.jpg', 0.2573], ['Harry Potter and the Chamber of Secrets (Book 2)', 'J. K. Rowling', 'http://images.amazon.com/images/P/0439064872.01.MZZZZZZZ.jpg', 0.2533]]